In [5]:
# ==============================================================================
# IMPORTS - Custom PF-PPO Implementation (Algorithm 1)
# ==============================================================================
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from dataclasses import dataclass
from typing import Dict, List
import random
import copy
import numpy as np
from tqdm.auto import tqdm

print("✓ Imports successful - Custom PF-PPO ready!")

# ==============================================================================
# MODEL LOADING
# ==============================================================================
BASE_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Important for batch generation in PPO

model = AutoModelForCausalLM.from_pretrained(
    BASE_NAME,
    torch_dtype=torch.bfloat16,  # ✅ Правильный параметр для from_pretrained()
    device_map="auto",
)

print(f"✓ Model loaded: {BASE_NAME}")
print(f"✓ Device: {model.device}")

# ==============================================================================
# MBPP DATASET
# ==============================================================================
MBPP_PATH = "/kaggle/input/asemjson/mbpp.jsonl" 

def load_mbpp(path: str):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            data.append(ex)
    return data

mbpp_all = load_mbpp(MBPP_PATH)
random.seed(42)
random.shuffle(mbpp_all)

train_data = mbpp_all[:350]
val_data   = mbpp_all[350:400]
test_data  = mbpp_all[400:500]

print(f"✓ Dataset loaded: {len(train_data)} train, {len(val_data)} val, {len(test_data)} test")

✓ Imports successful - Custom PF-PPO ready!


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
2025-12-27 15:35:56.776427: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766849757.259174      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766849757.378384      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766849758.469504      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766849758.469542      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766849758.469545      55

model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

✓ Model loaded: deepseek-ai/deepseek-coder-1.3b-instruct
✓ Device: cuda:0
✓ Dataset loaded: 350 train, 50 val, 100 test


In [6]:
K = 4  # число генераций на задачу

CODE_GEN_TEMPLATE = """You are an AI programming assistant, utilizing the
DeepSeek Coder model, developed by DeepSeek Company.
You only answer questions related to computer science.
You must respond with valid Python code only, without any explanations, comments, or markdown.

### Instruction
Write a Python function that solves the following problem:
{task_description}

Required function signature:
{signature}

Requirements:
- Implement only the core function.
- Include necessary import statements.
- Ensure the function signature matches the test cases.
- Write clean, efficient code.

### Response:
"""


def extract_signature(code: str) -> str:
    for line in code.splitlines():
        line = line.strip()
        if line.startswith("def "):
            return line  # например: "def sum_Even(l, r):"
    return ""  # на всякий случай

@dataclass
class MbppRLExample:
    task_id: int
    prompt: str
    test_code: str
    ref_code: str

class MbppRLDataset(Dataset):
    def __init__(self, examples: List[Dict]):
        self.examples = []
        for ex in examples:
            task_desc = ex["text"]
            tests = "\n".join(ex["test_list"])
            ref = ex["code"]
            signature = extract_signature(ref)

            prompt = CODE_GEN_TEMPLATE.format(
                task_description=task_desc,
                signature=signature,
            )

            self.examples.append(
                MbppRLExample(
                    task_id=ex.get("task_id", 0),
                    prompt=prompt,
                    test_code=tests,
                    ref_code=ref,
                )
            )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        return {
            "prompt": ex.prompt,
            "test_code": ex.test_code,
            "task_id": ex.task_id,
        }

train_dataset = MbppRLDataset(train_data)
val_dataset   = MbppRLDataset(val_data)
test_dataset   = MbppRLDataset(test_data)

def extract_code(completion: str) -> str:
    """
    Выделяет пригодный к выполнению Python-код из completion'а модели.
    Стратегия:
    1) Если есть блоки с ``````, берём тот, где есть 'def '.
    2) Иначе берём всё от первой строки, начинающейся с 'def '.
    3) fallback — исходный completion.
    """
    lines = completion.splitlines()

    # 1. Ищем блоки ``````
    if "```" in completion:
        parts = completion.split("```")
        for part in parts:
            if "def " in part:
                # убираем возможный язык после ```
                part_lines = part.splitlines()
                # отбрасываем первую строку, если это 'python' или пустота
                if part_lines and part_lines[0].strip().lower() in ("python", ""):
                    part_lines = part_lines[1:]
                return "\n".join(part_lines).strip()

    # 2. Иначе берём всё от первой строки с def
    for i, line in enumerate(lines):
        if line.strip().startswith("def "):
            return "\n".join(lines[i:]).strip()

    # 3. fallback
    return completion.strip()

In [7]:
import subprocess
import tempfile
import textwrap


def run_python_with_tests(solution_code: str, test_code: str, timeout: int = 5):
    """
    Выполняет solution_code + test_code в отдельном питоновском процессе и
    возвращает число прошедших и общее число тестов.
    Предполагается, что test_code использует assert'ы.
    """
    script = solution_code + "\n\n" + test_code
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(script)
        tmp_path = f.name

    try:
        proc = subprocess.run(
            ["python", tmp_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout,
            text=True,
        )
        # Грубая эвристика: считаем, что все assert прошли, если код возврата 0
        # и общее число тестов берём по количеству строк с "assert".
        total_tests = sum(1 for line in test_code.splitlines() if "assert" in line)
        passed = total_tests if proc.returncode == 0 else 0
    except subprocess.TimeoutExpired:
        passed, total_tests = 0, sum(1 for line in test_code.splitlines() if "assert" in line)
    finally:
        os.remove(tmp_path)

    return passed, total_tests

def reward_fn_test(prompts, completions, **kwargs):

    test_codes_batch = kwargs["test_code"]   # список строк длины batch_size
    task_ids_batch   = kwargs["task_id"]     # список int длины batch_size

    batch_size = len(prompts)
    num_generations = len(completions) // batch_size  # K

    expanded_test_codes = []
    expanded_task_ids = []
    for tcode, tid in zip(test_codes_batch, task_ids_batch):
        expanded_test_codes.extend([tcode] * num_generations)
        expanded_task_ids.extend([tid] * num_generations)

    rewards = []
    for completion, tid, tcode in zip(completions, expanded_task_ids, expanded_test_codes):
        code = extract_code(completion)
        passed, total = run_python_with_tests(code, tcode)
        r = passed / total if total > 0 else 0.0
        rewards.append(float(r))
    #print(rewards)
    return rewards 

In [8]:
# ==============================================================================
# PF-PPO TRAINING WITH R_test (Algorithm 1)
# ==============================================================================

# ------------------------------------------------------------------------------
# 1. Value Network (Critic) - Algorithm 1, line 1: "value network V_φ"
# ------------------------------------------------------------------------------
class ValueHead(nn.Module):
    """
    Critic network V_φ(s) that estimates the expected return for a given state.
    Used in Algorithm 1, line 10: L_value = 1/2(V_φ(A_i) - R_i)² + 1/2(V_φ(A_j) - R_j)²
    """
    def __init__(self, hidden_size=2048):
        super().__init__()
        self.value_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 1)
        )
    
    def forward(self, hidden_states):
        """
        Args:
            hidden_states: [batch_size, seq_len, hidden_size]
        Returns:
            values: [batch_size] - scalar value for each sequence
        """
        # Take last token's hidden state as sequence representation
        last_hidden = hidden_states[:, -1, :]  # [batch_size, hidden_size]
        return self.value_head(last_hidden).squeeze(-1)  # [batch_size]

# ------------------------------------------------------------------------------
# 2. PPO Configuration
# ------------------------------------------------------------------------------
@dataclass
class PPOConfig:
    """Hyperparameters for PF-PPO training (Algorithm 1)"""
    learning_rate: float = 1e-5        # α_θ and α_φ in Algorithm 1, line 1
    batch_size: int = 1                # Problems per step
    max_steps: int = 30               # Total training steps
    ppo_epochs: int = 1             # Multiple PPO updates per batch
    clip_range: float = 0.2            # ε for PPO clipping
    vf_coef: float = 0.5               # Value loss coefficient
    entropy_coef: float = 0.01         # β in Algorithm 1, line 11
    K: int = 4                         # Number of solutions per problem (Algorithm 1, line 3)
    delta: float = -0.1                 # Noise filtering threshold (Algorithm 1, line 6)
    max_new_tokens: int = 256
    temperature: float = 0.7
    top_p: float = 0.9
    kl_coef: float = 0.01              # KL divergence penalty vs reference model
    log_every: int = 10
    output_dir: str = "pf-ppo-test-checkpoints"

# ------------------------------------------------------------------------------
# 3. Helper Functions
# ------------------------------------------------------------------------------
@torch.no_grad()
def generate_solution(model, tokenizer, prompt, config):
    """Generate a single solution for a given prompt."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    
    gen_ids = model.generate(
        **inputs,
        max_new_tokens=config.max_new_tokens,
        temperature=config.temperature,
        top_p=config.top_p,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    
    # Extract only the generated part (without prompt)
    input_len = inputs["input_ids"].shape[1]
    generated_ids = gen_ids[0, input_len:]
    
    solution = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return solution

def compute_log_probs_and_value(model, value_head, tokenizer, prompt, solution):
    """
    Compute log probabilities and value estimate for a prompt+solution pair.
    """
    full_text = prompt + solution
    inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    
    # Forward pass through policy model
    outputs = model(**inputs, output_hidden_states=True)
    logits = outputs.logits  # [1, seq_len, vocab_size]
    hidden_states = outputs.hidden_states[-1]  # Last layer: [1, seq_len, hidden_size]
    
    # Compute log probabilities of the generated tokens
    # Shift logits and labels for next-token prediction
    shift_logits = logits[:, :-1, :].contiguous()  # [1, seq_len-1, vocab_size]
    shift_labels = inputs["input_ids"][:, 1:].contiguous()  # [1, seq_len-1]
    
    log_probs = F.log_softmax(shift_logits, dim=-1)  # [1, seq_len-1, vocab_size]
    
    # Get log prob of actual tokens
    token_log_probs = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)  # [1, seq_len-1]
    
    # Average log prob over generated tokens only
    prompt_len = len(tokenizer(prompt, truncation=True, max_length=2048)["input_ids"])
    gen_log_probs = token_log_probs[:, prompt_len-1:]  # Only solution tokens
    avg_log_prob = gen_log_probs.mean()
    
    # Compute value estimate
    value = value_head(hidden_states)  # [1]
    
    return avg_log_prob, value

@torch.no_grad()
def compute_ref_log_probs(ref_model, tokenizer, prompt, solution):
    """Compute log probs from reference model (for KL divergence)."""
    full_text = prompt + solution
    inputs = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=2048).to(ref_model.device)
    
    outputs = ref_model(**inputs)
    logits = outputs.logits
    
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = inputs["input_ids"][:, 1:].contiguous()
    
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_log_probs = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
    
    prompt_len = len(tokenizer(prompt, truncation=True, max_length=2048)["input_ids"])
    gen_log_probs = token_log_probs[:, prompt_len-1:]
    avg_log_prob = gen_log_probs.mean()
    
    return avg_log_prob

def compute_ppo_loss(
    model, ref_model, value_head, tokenizer, prompt, 
    solution_better, solution_worse, reward_better, reward_worse, config
):
    """
    PPO loss для пары (better, worse):
      - policy: парный advantage A = R_better - R_worse
      - value: MSE к их reward'ам
      - KL: простая траекторная регуляризация к ref_model
    """
    # ===== 1. Policy log-probs и value =====
    # better
    log_prob_better, value_better = compute_log_probs_and_value(
        model, value_head, tokenizer, prompt, solution_better
    )
    ref_log_prob_better = compute_ref_log_probs(
        ref_model, tokenizer, prompt, solution_better
    )

    # worse
    log_prob_worse, value_worse = compute_log_probs_and_value(
        model, value_head, tokenizer, prompt, solution_worse
    )
    ref_log_prob_worse = compute_ref_log_probs(
        ref_model, tokenizer, prompt, solution_worse
    )

    # ===== 2. Advantage для пары =====
    # A = R_better - R_worse, better получает +A, worse -A
    device = log_prob_better.device
    dtype  = log_prob_better.dtype

    adv = torch.tensor(reward_better - reward_worse, device=device, dtype=dtype)
    adv_better =  adv.detach()
    adv_worse  = -adv.detach()

    # ===== 3. PPO-клип по ratio (traj-wise) =====
    ratio_better = torch.exp(log_prob_better - ref_log_prob_better)
    ratio_worse  = torch.exp(log_prob_worse  - ref_log_prob_worse)

    clip_low, clip_high = 1.0 - config.clip_range, 1.0 + config.clip_range

    # better
    pg_better_unclipped = ratio_better * adv_better
    pg_better_clipped   = torch.clamp(ratio_better, clip_low, clip_high) * adv_better
    obj_better = torch.min(pg_better_unclipped, pg_better_clipped)

    # worse
    pg_worse_unclipped = ratio_worse * adv_worse
    pg_worse_clipped   = torch.clamp(ratio_worse, clip_low, clip_high) * adv_worse
    obj_worse = torch.min(pg_worse_unclipped, pg_worse_clipped)

    # хотим МАКСИМИЗИРОВАТЬ obj_better + obj_worse ⇒ ставим минус
    policy_loss = -(obj_better + obj_worse)

    # ===== 4. Value loss (как у тебя) =====
    target_better = torch.tensor([reward_better], device=device, dtype=value_better.dtype)
    target_worse  = torch.tensor([reward_worse],  device=device, dtype=value_worse.dtype)

    value_loss = 0.5 * F.mse_loss(value_better, target_better) \
               + 0.5 * F.mse_loss(value_worse,  target_worse)

    # ===== 5. KL-пенальти (без abs) =====
    # простой симметричный вариант по траекториям
    kl_better = (log_prob_better - ref_log_prob_better)
    kl_worse  = (log_prob_worse  - ref_log_prob_worse)
    kl_penalty = config.kl_coef * (kl_better**2 + kl_worse**2)

    # ===== 6. Суммарный loss =====
    total_loss = policy_loss + config.vf_coef * value_loss + kl_penalty

    return total_loss, {
        "policy_loss": policy_loss.item(),
        "value_loss": value_loss.item(),
        "kl_penalty": kl_penalty.item(),
    }

# ------------------------------------------------------------------------------
# 4. Main Training Loop (Algorithm 1)
# ------------------------------------------------------------------------------
def train_pf_ppo(model, value_head, train_dataset, config, reward_fn):
    """
    Algorithm 1: Pairwise Preference Proximal Policy Optimization (PF-PPO)
    
    Args:
        model: Policy network π_θ
        value_head: Value network V_φ
        train_dataset: MBPP training data
        config: PPOConfig
        reward_fn: Reward function (R_test or R_LLM)
    """
    print(f"\n{'='*80}")
    print(f"Starting PF-PPO Training (Algorithm 1)")
    print(f"{'='*80}")
    print(f"Config: K={config.K}, δ={config.delta}, steps={config.max_steps}")
    print(f"Learning rate: {config.learning_rate}, Clip: {config.clip_range}")
    
    # Initialize optimizers (Algorithm 1, line 1: learning rates α_θ, α_φ)
    optimizer_policy = AdamW(model.parameters(), lr=config.learning_rate)
    optimizer_value = AdamW(value_head.parameters(), lr=config.learning_rate)
    
    # Create reference model (frozen copy for KL divergence)
    ref_model = copy.deepcopy(model)
    ref_model.eval()
    for param in ref_model.parameters():
        param.requires_grad = False
    
    print(f"✓ Reference model created (frozen)")
    print(f"✓ Optimizers initialized")
    
    # Training loop
    model.train()
    value_head.train()
    
    step = 0
    pbar = tqdm(total=config.max_steps, desc="PF-PPO Training")
    
    while step < config.max_steps:
        # Algorithm 1, line 2: for each problem p in dataset
        problem_idx = np.random.randint(0, len(train_dataset))
        example = train_dataset[problem_idx]
        prompt = example["prompt"]
        test_code = example["test_code"]
        task_id = example["task_id"]
        
        # Algorithm 1, line 3: Generate K solutions
        model.eval()
        solutions = []
        rewards = []
        
        for _ in range(config.K):
            solution = generate_solution(model, tokenizer, prompt, config)
            solutions.append(solution)

        rewards = reward_fn(
            prompts=[prompt],                  # len = 1
            completions=solutions,             # len = K
            test_code=[test_code],             # len = 1
            task_id=[task_id],                 # len = 1
        )
        rewards = list(rewards)
        print(f"PROMPT:{prompt}", f"FIRST SOLUTION:{solutions[0]}")
        
        # Algorithm 1, lines 5-6: Create preference pairs with noise filtering
        pairs = []
        for i in range(config.K):
            for j in range(i + 1, config.K):
                # Algorithm 1, line 6: if |R_i - R_j| > δ (noise filtering)
                if abs(rewards[i] - rewards[j]) > config.delta:
                    # Algorithm 1, line 7: Determine preference
                    if rewards[i] > rewards[j]:
                        pairs.append((solutions[i], solutions[j], rewards[i], rewards[j]))
                    else:
                        pairs.append((solutions[j], solutions[i], rewards[j], rewards[i]))
        
        if len(pairs) == 0:
            # No valid pairs after filtering, skip this step
            pbar.update(1)
            step += 1
            continue
        
        # Algorithm 1, lines 8-14: PPO update for each pair
        model.train()
        value_head.train()
        
        total_loss_sum = 0.0
        policy_loss_sum = 0.0
        value_loss_sum = 0.0
        
        for sol_better, sol_worse, r_better, r_worse in pairs:
            # Multiple PPO epochs (line 8-14 repeated)
            for _ in range(config.ppo_epochs):
                # Algorithm 1, line 8-12: Compute losses
                loss, loss_dict = compute_ppo_loss(
                    model, ref_model, value_head, tokenizer, prompt,
                    sol_better, sol_worse, r_better, r_worse, config
                )
                
                # Algorithm 1, line 13-14: Update networks
                optimizer_policy.zero_grad()
                optimizer_value.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                torch.nn.utils.clip_grad_norm_(value_head.parameters(), 1.0)
                optimizer_policy.step()
                optimizer_value.step()
                
                total_loss_sum += loss.item()
                policy_loss_sum += loss_dict["policy_loss"]
                value_loss_sum += loss_dict["value_loss"]
        
        # Logging
        if step % config.log_every == 0:
            avg_reward = np.mean(rewards)
            num_pairs = len(pairs)
            pbar.set_postfix({
                "avg_reward": f"{avg_reward:.3f}",
                "pairs": num_pairs,
                "loss": f"{total_loss_sum/(num_pairs*config.ppo_epochs):.4f}"
            })
        
        pbar.update(1)
        step += 1
    
    pbar.close()
    print(f"\n{'='*80}")
    print(f"✓ PF-PPO Training Completed!")
    print(f"{'='*80}\n")
    
    return model, value_head

# ------------------------------------------------------------------------------
# 5. Initialize Value Head and Run Training
# ------------------------------------------------------------------------------
print("\n" + "="*80)
print("Initializing Value Network (Critic)")
print("="*80)

# Create value head
# Create value head (match model dtype!)
value_head = ValueHead(hidden_size=2048).to(model.device).to(torch.bfloat16)
print(f"✓ Value Head dtype: {next(value_head.parameters()).dtype}")
print(f"✓ Value Head created with {sum(p.numel() for p in value_head.parameters())} parameters")

# Create config
ppo_config = PPOConfig(
    learning_rate=3e-5,
    max_steps=100,
    K=4,
    delta=-0.1,
    ppo_epochs=1,
)


Initializing Value Network (Critic)
✓ Value Head dtype: torch.bfloat16
✓ Value Head created with 2099201 parameters


In [ ]:
# Train with R_test reward
print("\n" + "="*80)
print("Training PF-PPO with R_test (unit test rewards)")
print("="*80)

model_trained, value_head_trained = train_pf_ppo(
    model=model,
    value_head=value_head,
    train_dataset=train_dataset,
    config=ppo_config,
    reward_fn=reward_fn_test
)


Training PF-PPO with R_test (unit test rewards)

Starting PF-PPO Training (Algorithm 1)
Config: K=4, δ=-0.1, steps=100
Learning rate: 3e-05, Clip: 0.2
✓ Reference model created (frozen)
✓ Optimizers initialized


PF-PPO Training:   0%|          | 0/100 [00:00<?, ?it/s]

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

JUDGE_MODEL_NAME = "flowaicom/Flow-Judge-v0.1"

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME, use_fast=True)
judge_tokenizer.pad_token = judge_tokenizer.eos_token

judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
judge_model.eval()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [10]:
import re
from typing import List, Dict

JUDGE_PROMPT_TEMPLATE = """
You are an expert code reviewer evaluating the quality of Python solutions.
Analyze the provided code based on the following criteria:

Problem Statement:
{problem_description}

Generated Code:
{generated_code}

Evaluation Criteria:
1. Correctness: Does the code correctly solve the stated problem?
2. Code Quality: Is the code readable, well-structured, and following
   Python best practices?
3. Efficiency: Are algorithms and data structures chosen appropriately?
4. Completeness: Does the solution handle edge cases and include
   necessary imports?

Return ONLY the final score as floating point between 0.0 and 1.0
"""

@torch.no_grad()
def call_local_judge(problem_description: str, generated_code: str) -> float:
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        problem_description=problem_description,
        generated_code=generated_code,
    )

    inputs = judge_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(judge_model.device)

    output_ids = judge_model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        #temperature=0.0,
    )

    full_text = judge_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated = full_text[len(prompt):].strip()

    # ищем первое число вида 0.x, 1 или 1.0
    match = re.search(r"([01](?:\.\d+)?)", generated)
    if match:
        score = float(match.group(1))
        score = max(0.0, min(1.0, score))
    else:
        score = 0.0

    return score

In [11]:
def reward_llm_grpo_fn(prompts, completions, **kwargs):
    """
    prompts:     список исходных промптов (len = batch_size)
    completions: список сгенерированных ответов (len = batch_size * K)
    kwargs:      содержит те же метаданные, что и в датасете:
                 например, "task_description" или "prompt" целиком.
    """
    # достаём описание задачи; можно хранить отдельно, но можно и так:
    # здесь предполагаем, что в kwargs есть исходные тексты задач
    # если их нет — надо положить их в __getitem__ датасета,
    # например "task_description": ex.original_text
    task_desc_batch = kwargs.get("task_description", prompts)  # len = batch_size

    batch_size = len(prompts)
    num_generations = len(completions) // batch_size  # K

    # размножаем описания задач по числу генераций
    expanded_desc = []
    for desc in task_desc_batch:
        expanded_desc.extend([desc] * num_generations)

    rewards = []
    for raw_completion, desc in zip(completions, expanded_desc):
        code = extract_code(raw_completion)          # тот же extract_code, что для тестов
        r = call_local_judge(desc, code)            # R_LLM \in [0,1]
        rewards.append(float(r))

    return rewards

In [ ]:
model_trained_llm, value_head_trained_llm = train_pf_ppo(
    model=model,
    value_head=value_head,
    train_dataset=train_dataset,
    config=ppo_config,
    reward_fn=reward_llm_grpo_fn,     # LLM-judge reward
)


Starting PF-PPO Training (Algorithm 1)
Config: K=4, δ=-0.1, steps=100
Learning rate: 3e-05, Clip: 0.2
✓ Reference model created (frozen)
✓ Optimizers initialized


PF-PPO Training:   0%|          | 0/100 [00:00<?, ?it/s]

PROMPT:You are an AI programming assistant, utilizing the
DeepSeek Coder model, developed by DeepSeek Company.
You only answer questions related to computer science.
You must respond with valid Python code only, without any explanations, comments, or markdown.

### Instruction
Write a Python function that solves the following problem:
Write a function to compute the value of ncr%p.

Required function signature:
def ncr_modp(n, r, p):

Requirements:
- Implement only the core function.
- Include necessary import statements.
- Ensure the function signature matches the test cases.
- Write clean, efficient code.

### Response:
 FIRST SOLUTION:Here is a Python function that calculates the value of ncr%p using the formula ncr = n!/((n-r)!*r!)%p. This function is efficient and uses only basic arithmetic operations.

```python
def ncr_modp(n, r, p):
    if (r > n - r):
        return 0

    numerator = 1
    for i in range(n, n - r, -1):
        numerator = (numerator * i) % p

    denominator 

In [ ]:
import ast
import textwrap

def _norm(value, v_min, v_max, invert=False):
    """Линейная нормализация в [0,1]. invert=True => меньшее значение лучше."""
    if v_max == v_min:
        return 1.0
    x = (value - v_min) / (v_max - v_min)
    x = max(0.0, min(1.0, x))
    if invert:
        x = 1.0 - x
    return float(x)

class _SimpleComplexityVisitor(ast.NodeVisitor):
    def __init__(self):
        self.cc = 1
        self.max_depth = 0
        self._cur_depth = 0

    def _enter(self):
        self.cc += 1
        self._cur_depth += 1
        self.max_depth = max(self.max_depth, self._cur_depth)

    def _exit(self):
        self._cur_depth -= 1

    def _branch(self, node):
        self._enter()
        self.generic_visit(node)
        self._exit()

    def visit_If(self, node): self._branch(node)
    def visit_For(self, node): self._branch(node)
    def visit_AsyncFor(self, node): self._branch(node)
    def visit_While(self, node): self._branch(node)
    def visit_Try(self, node): self._branch(node)
    def visit_With(self, node): self._branch(node)
    def visit_AsyncWith(self, node): self._branch(node)

    def visit_BoolOp(self, node): self._branch(node)
    def visit_IfExp(self, node): self._branch(node)

def _analyze_simple(code: str):
    """
    Возвращает:
      {
        "cc": ...,
        "nd": ...,
        "loc": ...,
        "comment_ratio": ...,
      }
    """
    code = textwrap.dedent(code)
    lines = code.splitlines()

    nonblank = [ln for ln in lines if ln.strip() != ""]
    comment_lines = [ln for ln in nonblank if ln.lstrip().startswith("#")]
    code_lines = [ln for ln in nonblank if not ln.lstrip().startswith("#")]

    loc = len(code_lines)
    total_nonblank = len(nonblank)
    comment_ratio = (
        len(comment_lines) / total_nonblank if total_nonblank > 0 else 0.0
    )

    try:
        tree = ast.parse(code)
    except SyntaxError:
        # считаем максимально сложным и без комментариев
        return {
            "cc": 20,
            "nd": 5,
            "loc": max(loc, 120),
            "comment_ratio": comment_ratio,
        }

    visitor = _SimpleComplexityVisitor()
    visitor.visit(tree)
    cc = visitor.cc
    nd = visitor.max_depth

    return {
        "cc": cc,
        "nd": nd,
        "loc": max(loc, 1),
        "comment_ratio": comment_ratio,
    }

def compute_static_reward_simple(code: str) -> float:
    """
    Упрощённый proxy static reward в [0,1] для коротких решений.
    """
    stats = _analyze_simple(code)

    cc = stats["cc"]
    nd = stats["nd"]
    loc = stats["loc"]
    comment_ratio = stats["comment_ratio"]

    # Границы нормализации под MBPP‑подобные задачи [web:210][web:180].
    s_cc  = _norm(cc,  1, 12, invert=True)   # 1–12 ветвлений
    s_nd  = _norm(nd,  0,  4, invert=True)   # до 4 уровней вложенности
    s_loc = _norm(loc, 3,  60, invert=True)  # 3–60 строк кода

    s_complexity = (s_cc + s_nd + s_loc) / 3.0
    s_readability = max(0.0, min(1.0, float(comment_ratio)))

    r_static = 0.6 * s_complexity + 0.4 * s_readability
    return max(0.0, min(1.0, float(r_static)))

In [ ]:
@torch.no_grad()
def evaluate_model(
    model,
    tokenizer,
    dataset,
    max_new_tokens: int = 256,
):
    model.eval()
    task_results = []

    for ex in dataset:
        prompt    = ex["prompt"]
        test_code = ex["test_code"]
        task_id   = ex["task_id"]

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]

        gen_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

        completion = tokenizer.decode(
            gen_ids[0, input_len:], skip_special_tokens=True
        )
        code = extract_code(completion)

        # R_test
        passed, total = run_python_with_tests(code, test_code)
        r_test = passed / total if total > 0 else 0.0

        # R_static_simple
        r_static = compute_static_reward_simple(code)
            

        task_results.append(
            {
                "task_id": task_id,
                "r_test": r_test,
                "r_static": r_static,
            }
        )

    mean_r_test = sum(r["r_test"] for r in task_results) / len(task_results)
    mean_r_static = sum(r["r_static"] for r in task_results) / len(task_results)
    r_final = 2 * mean_r_test * mean_r_static / (mean_r_test+mean_r_static) 

    return {
        "mean_r_test": mean_r_test,
        "mean_r_static": mean_r_static,
        "mean_r_final_05": r_final,
        "per_task": task_results,
    }

In [ ]:
evaluate_model(model_trained_llm, tokenizer, test_dataset)